# Convolutional Neural Network Code

## TODO (might be outdated)
- Create a bespoke CNN model for this data set
- Think about cropping the image to only include the road maybe but I guess the image has to be square
- Think of how much padding should be applied to this CNN based on how important the borders of the image are, consult this article (https://medium.com/thedeephub/convolutional-neural-networks-a-comprehensive-guide-5cc0b5eae175) and think about the stride too.
- Will probably need to add padding 
- Choose a method to normalize the data you have which is helpful for YOUR data


Loading libraries in the code cell below:

In [1]:
# from tensorflow.keras import datasets, layers, models
# import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

Setup vars:

In [3]:
data_dir = "../Data/cnn-classes"
batch_size = 16
num_epochs = 10
learning_rate = 1e-4
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15
seed = 42

Checking for GPU and ensuring that the model is run on it:

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


Loading pretrained weights:

In [5]:
weights = ResNet18_Weights.DEFAULT

Transforming data for the model:

# todo: Maybe use the whole images instead???

In [6]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights.transforms().mean,
        std=weights.transforms().std
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights.transforms().mean,
        std=weights.transforms().std
    )
])

full_dataset = datasets.ImageFolder(root=data_dir)
print("Classes:", full_dataset.classes)
print("Total images:", len(full_dataset))

Classes: ['optimal_laps', 'standard_laps', 'sub_standard_laps']
Total images: 488


Defining model:

In [7]:
total_size = len(full_dataset)
train_size = int(train_ratio * total_size)
val_size = int(val_ratio * total_size)
test_size = total_size - train_size - val_size

generator = torch.Generator().manual_seed(seed)

train_subset, val_subset, test_subset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=generator
)

train_subset.dataset = datasets.ImageFolder(root=data_dir, transform=train_transform)
val_subset.dataset = datasets.ImageFolder(root=data_dir, transform=eval_transform)
test_subset.dataset = datasets.ImageFolder(root=data_dir, transform=eval_transform)

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=0)

model = models.resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, len(full_dataset.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


Defining functions to run the models:

In [8]:

def run_epoch(model, loader, criterion, optimizer=None):
    if optimizer is None:
        model.eval()
    else:
        model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.set_grad_enabled(optimizer is not None):
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            if optimizer is not None:
                optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            if optimizer is not None:
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

Running the model:

In [9]:

best_val_acc = 0.0
best_model_path = "best_resnet18.pth"

for epoch in range(num_epochs):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)

print("Best validation accuracy:", best_val_acc)

model.load_state_dict(torch.load(best_model_path, map_location=device))
test_loss, test_acc = run_epoch(model, test_loader, criterion)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

Epoch 1/10 | Train Loss: 1.2775 | Train Acc: 0.3372 | Val Loss: 1.2000 | Val Acc: 0.4932
Epoch 2/10 | Train Loss: 0.9098 | Train Acc: 0.5748 | Val Loss: 1.0277 | Val Acc: 0.5205
Epoch 3/10 | Train Loss: 0.7313 | Train Acc: 0.6950 | Val Loss: 1.0561 | Val Acc: 0.4795
Epoch 4/10 | Train Loss: 0.6157 | Train Acc: 0.7713 | Val Loss: 1.1497 | Val Acc: 0.4658
Epoch 5/10 | Train Loss: 0.4523 | Train Acc: 0.8446 | Val Loss: 1.3126 | Val Acc: 0.4110
Epoch 6/10 | Train Loss: 0.3882 | Train Acc: 0.8739 | Val Loss: 1.2560 | Val Acc: 0.4521
Epoch 7/10 | Train Loss: 0.3028 | Train Acc: 0.9032 | Val Loss: 1.4021 | Val Acc: 0.4658
Epoch 8/10 | Train Loss: 0.2075 | Train Acc: 0.9238 | Val Loss: 1.5267 | Val Acc: 0.5068
Epoch 9/10 | Train Loss: 0.1692 | Train Acc: 0.9560 | Val Loss: 1.6589 | Val Acc: 0.4247
Epoch 10/10 | Train Loss: 0.1989 | Train Acc: 0.9267 | Val Loss: 1.7168 | Val Acc: 0.4384
Best validation accuracy: 0.5205479452054794
Test Loss: 1.0970
Test Accuracy: 0.4595
